In [3]:
import joblib
import json
import pandas as pd
import numpy as np

# 1. GEREKLİ DOSYALARI YÜKLE
model = joblib.load('personel_ai_model.pkl')
target_le = joblib.load('target_encoder.pkl')
# feature_encoders bu aşamada hata verebildiği için manuel sayısal giriş yapacağız

# Bilgi bankasını yükle
with open('program_icerikleri.json', 'r', encoding='utf-8') as f:
    bilgi_bankasi = json.load(f)

# 2. TEST İÇİN ÖRNEK KULLANICI VERİSİ (Sayısal Karşılıklarla)
# Not: Eğitimde Cinsiyet: Male=0, Female=1 şeklinde kodlanmıştı.
# Sakatlik_Bolgesi, Hedef ve Ekipman değerlerini de eğitimdeki karşılıklarına göre giriyoruz.

kisi_verisi = {
    'Yas': 25,
    'Cinsiyet': 0,             # Male -> 0
    'Boy': 180,
    'Kilo': 85,
    'Aktivite_Seviyesi': 1.55,
    'Sakatlik_Bolgesi': 4,     # 'Yok' -> Genelde son alfabetik sıradadır (0-4 arası kontrol et)
    'Sakatlik_Siddeti': 0.0,
    'Spor_Gecmisi_Yil': 2,
    'Hedef': 0,                # 'Güç ve Kondisyon'
    'Ekipman': 1,              # 'Full Gym'
    'Ogun_Tercihi': 3,
    'VKI': 26.23
}

# DataFrame oluştur (Sütun sırası eğitimle birebir aynı olmalı)
df_test = pd.DataFrame([kisi_verisi])

# 3. MODEL TAHMİNİ YAP
tahmin_indeksi = model.predict(df_test)[0]
program_etiketi = target_le.inverse_transform([tahmin_indeksi])[0]

# 4. KALORİ HESAPLAMA (Mifflin-St Jeor)
# Cinsiyet 0 (Male) ise formül:
bmr = 10 * kisi_verisi['Kilo'] + 6.25 * kisi_verisi['Boy'] - 5 * kisi_verisi['Yas'] + 5
gunluk_kalori = int(bmr * kisi_verisi['Aktivite_Seviyesi'])

# 5. SONUÇLARI EKRANA YAZDIR
print("="*45)
print("     KİŞİSEL YAPAY ZEKA ASİSTANI ANALİZİ     ")
print("="*45)
print(f"Saptanan Profil: {program_etiketi}")
print(f"Günlük Alman Gereken Kalori: {gunluk_kalori} kcal")
print("-" * 45)

if program_etiketi in bilgi_bankasi:
    detay = bilgi_bankasi[program_etiketi]
    print(f"Önerilen Program: {detay['Antrenman_Adi']}")
    print(f"\nPazartesi Programı:")
    print(f"  - Odak Bölge: {detay['Pazartesi']['Odak']}")
    print(f"  - Egzersizler: {', '.join(detay['Pazartesi']['Hareketler'])}")
    print(f"  - Beslenme Tipi: {detay['Pazartesi']['Beslenme_Tipi']}")
else:
    print(f"Bilgi: '{program_etiketi}' için JSON detayları bulunamadı.")
print("="*45)

     KİŞİSEL YAPAY ZEKA ASİSTANI ANALİZİ     
Saptanan Profil: PPL_SPLIT
Günlük Alman Gereken Kalori: 2875 kcal
---------------------------------------------
Önerilen Program: Push-Pull-Legs (İtme-Çekme-Bacak)

Pazartesi Programı:
  - Odak Bölge: İtme (Göğüs, Omuz, Triceps)
  - Egzersizler: Bench Press 4x10, Overhead Press 3x12, Triceps Pushdown 3x15
  - Beslenme Tipi: Yüksek Karbonhidrat ve Protein
